**IMPORTANT: This project was built on and for Google Colab. Running in outside environments may result in need for extra tweaks and errors.**

In [ ]:
!unzip -o medgemma_lora_model.zip

Archive:  medgemma_lora_model.zip
   creating: lora_model/
  inflating: lora_model/tokenizer.json  
  inflating: lora_model/tokenizer_config.json  
  inflating: lora_model/preprocessor_config.json  
  inflating: lora_model/special_tokens_map.json  
  inflating: lora_model/tokenizer.model  
  inflating: lora_model/chat_template.jinja  
  inflating: lora_model/adapter_model.safetensors  
  inflating: lora_model/processor_config.json  
  inflating: lora_model/README.md    
  inflating: lora_model/adapter_config.json  
 extracting: lora_model/added_tokens.json  


Environment setup

Install needed dependencies and importing keys

To run this project properly you need 1. A functioning Gemini key 2. valid qdrant key and cluster API 3. a GPU with CUDA enabled and 4. Valid Huggingface token

In [ ]:
!pip install -q google-genai qdrant-client sentence-transformers transformers accelerate bitsandbytes pymupdf langchain langchain-community langchain_google_genai
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130
!pip install ragas pandas asyncio
!pip install colpali-engine pdf2image
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers peft accelerate bitsandbytes
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130

import qdrant_client
import os
from google.colab import userdata
from unsloth import FastLanguageModel

os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
os.environ["QDRANT_API_KEY"] = userdata.get('QDRANT_API_KEY')
os.environ["QDRANT_ENDPOINT_URL"] = userdata.get('QDRANT_ENDPOINT_URL')
os.environ["HUGGINGFACETOKEN"] = userdata.get('HUGGINGFACETOKEN')
os.environ["HF_TOKEN"] = userdata.get('HUGGINGFACETOKEN')
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


KeyboardInterrupt: 

CPU VERIFICATION

Vestige from local running; optional for Colab

In [ ]:
import torch

gpu_available = torch.cuda.is_available()
print(f"GPU Available: {gpu_available}")

if gpu_available:
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    x = torch.rand(3, 3).cuda()
    print("\nTensor successfully created on GPU:")
    print(x)
else:
    print(" CUDA is NOT available.")

GPU Available: True
Device Name: NVIDIA A100-SXM4-40GB
CUDA Version: 12.8

Tensor successfully created on GPU:
tensor([[0.1539, 0.4303, 0.9908],
        [0.0979, 0.0939, 0.2206],
        [0.6426, 0.1171, 0.6559]], device='cuda:0')


Config and file creation

**UPLOAD THE MAIN DATASET IN /content/data/ BEFORE RUNNING THIS**

**UPLOAD THE KNOWLEDGE DATASET IN /content/knowledge_data/ BEFORE RUNNING THIS**





In [ ]:
import os

#create Directories
base_dir = "/content"
data_dir = os.path.join(base_dir, "data")
text_output_dir = os.path.join(base_dir, "text_output")
text_output_chunked_dir = os.path.join(base_dir, "text_output_chunked")
knowledge_data_dir = os.path.join(base_dir, "knowledge_data")
knowledge_text_output_dir = os.path.join(base_dir, "knowledge_text_output")
knowledge_text_output_chunked_dir = os.path.join(base_dir, "knowledge_text_output_chunked")

os.makedirs(data_dir, exist_ok=True)
os.makedirs(text_output_dir, exist_ok=True)
os.makedirs(text_output_chunked_dir, exist_ok=True)
os.makedirs(knowledge_data_dir, exist_ok=True)
os.makedirs(knowledge_text_output_dir, exist_ok=True)
os.makedirs(knowledge_text_output_chunked_dir, exist_ok=True)

print(f"Directories created at: {base_dir}")

# create config.py file
config_content = """
import os
import logging

# Logger Configuration
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# HUGGINGFACE CONFIGURATION
HUGGINGFACETOKEN = os.environ.get("HUGGINGFACETOKEN")

# QDRANT CLOUD CONFIGURATION
QDRANT_API_KEY = os.environ.get("QDRANT_API_KEY")
QDRANT_ENDPOINT_URL = os.environ.get("QDRANT_ENDPOINT_URL")

# GEMINI CONFIGURATION
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
"""

with open("config.py", "w") as f:
    f.write(config_content)

print("config.py created.")

Directories created at: /content
config.py created.


Text extraction (seperate for main and knowledge)

In [ ]:
import fitz
import os
import json
import re

knowledge_data_folder = "/content/knowledge_data"
knowledge_output_folder = "/content/knowledge_text_output"

os.makedirs(knowledge_output_folder, exist_ok=True)

knowledge_all_pdf_files = [os.path.join(knowledge_data_folder, f) for f in os.listdir(knowledge_data_folder) if f.lower().endswith('.pdf')]

print(f"Found {len(knowledge_all_pdf_files)} PDF files to process.")

def cleanup_text(text):
    # regex stuff
    text = re.sub(r'\n+', '\n', text)
    text = '\n'.join(line.strip() for line in text.splitlines())
    text = re.sub(r'^\d+$|^\w$', '', text, flags=re.MULTILINE)
    text = re.sub(r' +', ' ', text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'-\s+', '', text)
    return text.strip()

for pdf_path in knowledge_all_pdf_files:
    try:
        print(f"Attempting to open: {pdf_path}")
        doc = fitz.open(pdf_path)

        if doc.is_encrypted:
            print(f"  PDF '{os.path.basename(pdf_path)}' is encrypted. Skipping.")
            doc.close()
            continue

        base_filename = os.path.splitext(os.path.basename(pdf_path))[0]
        output_filename = f"{base_filename}.json"
        output_path = os.path.join(knowledge_output_folder, output_filename)

        pdf_data = {
            "filename": os.path.basename(pdf_path),
            "pages": []
        }

        print(f"Extracting text from: {os.path.basename(pdf_path)}")
        for i, page in enumerate(doc):
            try:
                text = page.get_text()
                cleaned_text = cleanup_text(text)
                pdf_data["pages"].append({
                    "page_number": i + 1,
                    "text": cleaned_text
                })
            except Exception as page_e:
                print(f"  Error extracting text from page {i+1} in {os.path.basename(pdf_path)}: {page_e}")
                pdf_data["pages"].append({
                    "page_number": i + 1,
                    "text": f"Error extracting text: {str(page_e)}"
                })
                continue

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(pdf_data, f, ensure_ascii=False, indent=4)

        doc.close()
        print(f"  Saved text from {os.path.basename(pdf_path)} to {output_filename}")

    except Exception as e:
        print(f"Error processing {os.path.basename(pdf_path)}: {e}")

print("\nText extraction complete.")

Found 40 PDF files to process.
Attempting to open: /content/knowledge_data/Peter Lydyard_Michael Cole_J B Holton_William L Irving_Nino Por - Case studies in infectious disease (2010, Taylor & Francis) - libgen.li (1)-288-299.pdf
Extracting text from: Peter Lydyard_Michael Cole_J B Holton_William L Irving_Nino Por - Case studies in infectious disease (2010, Taylor & Francis) - libgen.li (1)-288-299.pdf
  Saved text from Peter Lydyard_Michael Cole_J B Holton_William L Irving_Nino Por - Case studies in infectious disease (2010, Taylor & Francis) - libgen.li (1)-288-299.pdf to Peter Lydyard_Michael Cole_J B Holton_William L Irving_Nino Por - Case studies in infectious disease (2010, Taylor & Francis) - libgen.li (1)-288-299.json
Attempting to open: /content/knowledge_data/Peter Lydyard_Michael Cole_J B Holton_William L Irving_Nino Por - Case studies in infectious disease (2010, Taylor & Francis) - libgen.li (1)-12-27.pdf
Extracting text from: Peter Lydyard_Michael Cole_J B Holton_William L

In [ ]:
import fitz
import os
import json
import re

data_folder = "/content/data"
output_folder = "/content/text_output"

os.makedirs(output_folder, exist_ok=True)

all_pdf_files = [os.path.join(data_folder, f) for f in os.listdir(data_folder) if f.lower().endswith('.pdf')]

print(f"Found {len(all_pdf_files)} PDF files to process.")

def cleanup_text(text):
    # regex stuff
    text = re.sub(r'\n+', '\n', text)
    text = '\n'.join(line.strip() for line in text.splitlines())
    text = re.sub(r'^\d+$|^\w$', '', text, flags=re.MULTILINE)
    text = re.sub(r' +', ' ', text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'-\s+', '', text)
    return text.strip()

for pdf_path in all_pdf_files:
    try:
        print(f"Attempting to open: {pdf_path}")
        doc = fitz.open(pdf_path)

        if doc.is_encrypted:
            print(f"  PDF '{os.path.basename(pdf_path)}' is encrypted. Skipping.")
            doc.close()
            continue

        base_filename = os.path.splitext(os.path.basename(pdf_path))[0]
        output_filename = f"{base_filename}.json"
        output_path = os.path.join(output_folder, output_filename)

        pdf_data = {
            "filename": os.path.basename(pdf_path),
            "pages": []
        }

        print(f"Extracting text from: {os.path.basename(pdf_path)}")
        for i, page in enumerate(doc):
            try:
                text = page.get_text()
                cleaned_text = cleanup_text(text)
                pdf_data["pages"].append({
                    "page_number": i + 1,
                    "text": cleaned_text
                })
            except Exception as page_e:
                print(f"  Error extracting text from page {i+1} in {os.path.basename(pdf_path)}: {page_e}")
                pdf_data["pages"].append({
                    "page_number": i + 1,
                    "text": f"Error extracting text: {str(page_e)}"
                })
                continue

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(pdf_data, f, ensure_ascii=False, indent=4)

        doc.close()
        print(f"  Saved text from {os.path.basename(pdf_path)} to {output_filename}")

    except Exception as e:
        print(f"Error processing {os.path.basename(pdf_path)}: {e}")

print("\nText extraction complete.")

Found 93 PDF files to process.
Attempting to open: /content/data/66---A-32-Year-Old-Man-from-Malawi-With-Pain-in-the-_2022_Clinical-Cases-in-.pdf
Extracting text from: 66---A-32-Year-Old-Man-from-Malawi-With-Pain-in-the-_2022_Clinical-Cases-in-.pdf
  Saved text from 66---A-32-Year-Old-Man-from-Malawi-With-Pain-in-the-_2022_Clinical-Cases-in-.pdf to 66---A-32-Year-Old-Man-from-Malawi-With-Pain-in-the-_2022_Clinical-Cases-in-.json
Attempting to open: /content/data/50---A-24-Year-Old-Man-of-Turkish-Origin-With-Jau_2022_Clinical-Cases-in-Tro.pdf
Extracting text from: 50---A-24-Year-Old-Man-of-Turkish-Origin-With-Jau_2022_Clinical-Cases-in-Tro.pdf
  Saved text from 50---A-24-Year-Old-Man-of-Turkish-Origin-With-Jau_2022_Clinical-Cases-in-Tro.pdf to 50---A-24-Year-Old-Man-of-Turkish-Origin-With-Jau_2022_Clinical-Cases-in-Tro.json
Attempting to open: /content/data/42---A-41-Year-Old-Male-Traveller-Returning-from-Au_2022_Clinical-Cases-in-T.pdf
Extracting text from: 42---A-41-Year-Old-Male-Trav

Structured Metadata Extraction

For Knowledge

In [ ]:
import os
import json
import time
from pathlib import Path
from google import genai


knowledge_INPUT_DIR = Path("/content/knowledge_text_output")
knowledge_OUTPUT_DIR = Path("/content/knowledge_text_output_chunked")

try:
    API_KEY = os.environ["GEMINI_API_KEY"]
except KeyError:
    print("!! ERROR: GEMINI_API_KEY not set.")
    raise

MODEL_NAME = "gemini-2.5-pro"

# delay in seconds between API calls just to prevent rate limit if relevant
REQUEST_DELAY_SEC = 1


SYSTEM_PROMPT = """## System
You are **TropID-Extractor**, an expert clinical information extractor for **tropical & infectious diseases**.
Your task: **read a free-text clinical case in JSON format** and return a **single JSON object** where **each section is one coherent full-text block** (no bulletizing into tiny subfields).
If a section is not present, set it to `null`. **Do not guess.** If there are irrelevant info, they can be left out.
**Output ONLY valid JSON** — no preamble, no commentary.
### Formatting & Safety Rules
- **One JSON object only.**
- **Full-text blocks:** Each field below must be a **cohesive paragraph** (or short multi-sentence block) stitched from the case text; paraphrase minimally, preserve clinical meaning, and **do not invent** missing details.
- **Attribution discipline:** Prefer exact phrases from the source for key facts (fever pattern, exposures, test names, titers) but keep the prose readable.
- **Units & names:** Keep units and proper names as written (°C/°F, NS1, thick smear, RDT, species, titers/CT values).
- **Privacy:** Exclude any direct identifiers if present.
- **Uncertainty:** If the case explicitly says something is “unclear/unknown,” include that wording.
- **Final diagnosis:** Write a concise paragraph that **states the diagnosis, the causative agent if given, and the evidence** (labs/imaging/epidemiology/response to therapy).
- **Disease name (short):** After `final_diagnosis`, fill `disease_name_short` with the **best disease name only** (e.g., “Dengue fever”, “Falciparum malaria”, “Scrub typhus”).
---
Return **only** this JSON schema (exact keys, same order):
```json
{
  "patient_information": null,
  "chief_complaint": null,
  "history_of_present_illness": null,
  "exposure_and_epidemiology": null,
  "vitals": null,
  "physical_exam": null,
  "labs_and_diagnostics": null,
  "differential_diagnosis": null,
  "management_and_clinical_course": null,
  "final_diagnosis": null,
  "disease_name_short": null
}
```

### Field guidance (concise)
- **patient_information**: age/sex; relevant comorbidities/immunosuppression; vaccination/allergy info if stated.
- **chief_complaint**: one-line problem + duration.
- **history_of_present_illness**: timeline, key symptoms, pertinent negatives, severity pattern.
- **exposure_and_epidemiology**: residence/travel (place/setting, dates if present), vectors (mosquito/tick), animals, water/food risks, contacts, season/outbreak context, occupation.
- **vitals**: all reported vital signs as text (fever values, BP, HR, RR, SpO₂).
- **physical_exam**: salient systems (skin, HEENT, chest, abdo, neuro, lymph, etc.).
- **labs_and_diagnostics**: CBC trends, key chem/coag, inflammatory markers, microbiology/serology/PCR (assay + result + titer/CT if present), malaria tests, imaging highlights.
- **differential_diagnosis**: succinct narrative of the main alternatives considered, with one-sentence justification for/against each.
- **management_and_clinical_course**: antimicrobials (drug/dose if given), supportive care, procedures; response, complications, outcome.
- **final_diagnosis**: 3–5 sentences: explicit disease name ± causative agent, confirmation method (e.g., NS1+, thick smear species, PCR/serology), and why alternatives were ruled out.
- **disease_name_short**: the disease name only (no agent, no method).
"""

def process_single_file(client, input_path, output_path):
    """
    Reads one JSON file, sends its content to the Gemini API,
    and saves the structured JSON response.
    """
    try:
        # read the source JSON file
        with open(input_path, 'r', encoding='utf-8') as f:
            source_data = json.load(f)
        # convert the source JSON object back into a string to feed to the model
        case_data_string = json.dumps(source_data, indent=2)
        # combine SYSTEM_PROMPT with case data
        full_query = f"{SYSTEM_PROMPT}\n\n---\n\nExtract information from the following clinical case:\n{case_data_string}"

    except json.JSONDecodeError:
        print(f"   ... ERROR: Failed to decode source JSON. Skipping.")
        return
    except Exception as e:
        print(f"   ... ERROR reading file: {e}. Skipping.")
        return

    try:
        config_dict = {
            "response_mime_type": "application/json"
        }

        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=[full_query],
            config=config_dict
        )

        # the API is already forced to return JSON, clean it just in case
        # just in case it includes markdown backticks
        cleaned_response_text = response.text.strip().replace("```json", "").replace("```", "")

        try:
            output_json = json.loads(cleaned_response_text)
        except json.JSONDecodeError:
            print(f"   ... ERROR: API did not return valid JSON. Response was:")
            print(cleaned_response_text)
            return

        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(output_json, f, indent=2)

        print(f"   ... Success, Saved to {output_path.name}")

    except Exception as e:
        print(f"   ... ERROR during API call: {e}")

def main():
    # batch processor
    print(f"Input folder:  {knowledge_INPUT_DIR}")
    print(f"Output folder: {knowledge_OUTPUT_DIR}")
    print(f"Model:         {MODEL_NAME}")
    print("-" * 40)

    knowledge_OUTPUT_DIR.mkdir(exist_ok=True)
    client = genai.Client(api_key=API_KEY)

    # grab all .json files from the input directory
    input_files = list(knowledge_INPUT_DIR.glob("*.json"))

    if not input_files:
        print(f"No .json files found in {knowledge_INPUT_DIR}. Exiting.")
        return

    total_files = len(input_files)
    print(f"Found {total_files} JSON files to process.")

    for i, input_path in enumerate(input_files):
        print(f"\n[{i+1}/{total_files}] Processing: {input_path.name}")
        output_path = knowledge_OUTPUT_DIR / input_path.name

        # check if output already exists
        if output_path.exists():
            print("   ... Output file already exists. Skipping.")
            continue

        # process the file
        process_single_file(client, input_path, output_path)

        # rate limit
        if i < total_files - 1:
            time.sleep(REQUEST_DELAY_SEC)

    print("\n" + "-" * 40)
    print("Batch processing complete.")

if __name__ == "__main__":
    main()


Input folder:  /content/knowledge_text_output
Output folder: /content/knowledge_text_output_chunked
Model:         gemini-2.5-pro
----------------------------------------
Found 40 JSON files to process.

[1/40] Processing: Peter Lydyard_Michael Cole_J B Holton_William L Irving_Nino Por - Case studies in infectious disease (2010, Taylor & Francis) - libgen.li (1)-450-461.json
   ... Success, Saved to Peter Lydyard_Michael Cole_J B Holton_William L Irving_Nino Por - Case studies in infectious disease (2010, Taylor & Francis) - libgen.li (1)-450-461.json

[2/40] Processing: Peter Lydyard_Michael Cole_J B Holton_William L Irving_Nino Por - Case studies in infectious disease (2010, Taylor & Francis) - libgen.li (1)-326-338.json
   ... Success, Saved to Peter Lydyard_Michael Cole_J B Holton_William L Irving_Nino Por - Case studies in infectious disease (2010, Taylor & Francis) - libgen.li (1)-326-338.json

[3/40] Processing: Peter Lydyard_Michael Cole_J B Holton_William L Irving_Nino Por - C

For Main

In [ ]:
import os
import json
import time
from pathlib import Path
from google import genai


INPUT_DIR = Path("/content/text_output")
OUTPUT_DIR = Path("/content/text_output_chunked")

try:
    API_KEY = os.environ["GEMINI_API_KEY"]
except KeyError:
    print("!! ERROR: GEMINI_API_KEY not set.")
    raise

MODEL_NAME = "gemini-2.5-pro"

# delay in seconds between API calls just to prevent rate limit if relevant
REQUEST_DELAY_SEC = 1


SYSTEM_PROMPT = """## System
You are **TropID-Extractor**, an expert clinical information extractor for **tropical & infectious diseases**.
Your task: **read a free-text clinical case in JSON format** and return a **single JSON object** where **each section is one coherent full-text block** (no bulletizing into tiny subfields).
If a section is not present, set it to `null`. **Do not guess.** If there are irrelevant info, they can be left out.
**Output ONLY valid JSON** — no preamble, no commentary.
### Formatting & Safety Rules
- **One JSON object only.**
- **Full-text blocks:** Each field below must be a **cohesive paragraph** (or short multi-sentence block) stitched from the case text; paraphrase minimally, preserve clinical meaning, and **do not invent** missing details.
- **Attribution discipline:** Prefer exact phrases from the source for key facts (fever pattern, exposures, test names, titers) but keep the prose readable.
- **Units & names:** Keep units and proper names as written (°C/°F, NS1, thick smear, RDT, species, titers/CT values).
- **Privacy:** Exclude any direct identifiers if present.
- **Uncertainty:** If the case explicitly says something is “unclear/unknown,” include that wording.
- **Final diagnosis:** Write a concise paragraph that **states the diagnosis, the causative agent if given, and the evidence** (labs/imaging/epidemiology/response to therapy).
- **Disease name (short):** After `final_diagnosis`, fill `disease_name_short` with the **best disease name only** (e.g., “Dengue fever”, “Falciparum malaria”, “Scrub typhus”).
---
Return **only** this JSON schema (exact keys, same order):
```json
{
  "patient_information": null,
  "chief_complaint": null,
  "history_of_present_illness": null,
  "exposure_and_epidemiology": null,
  "vitals": null,
  "physical_exam": null,
  "labs_and_diagnostics": null,
  "differential_diagnosis": null,
  "management_and_clinical_course": null,
  "final_diagnosis": null,
  "disease_name_short": null
}
```

### Field guidance (concise)
- **patient_information**: age/sex; relevant comorbidities/immunosuppression; vaccination/allergy info if stated.
- **chief_complaint**: one-line problem + duration.
- **history_of_present_illness**: timeline, key symptoms, pertinent negatives, severity pattern.
- **exposure_and_epidemiology**: residence/travel (place/setting, dates if present), vectors (mosquito/tick), animals, water/food risks, contacts, season/outbreak context, occupation.
- **vitals**: all reported vital signs as text (fever values, BP, HR, RR, SpO₂).
- **physical_exam**: salient systems (skin, HEENT, chest, abdo, neuro, lymph, etc.).
- **labs_and_diagnostics**: CBC trends, key chem/coag, inflammatory markers, microbiology/serology/PCR (assay + result + titer/CT if present), malaria tests, imaging highlights.
- **differential_diagnosis**: succinct narrative of the main alternatives considered, with one-sentence justification for/against each.
- **management_and_clinical_course**: antimicrobials (drug/dose if given), supportive care, procedures; response, complications, outcome.
- **final_diagnosis**: 3–5 sentences: explicit disease name ± causative agent, confirmation method (e.g., NS1+, thick smear species, PCR/serology), and why alternatives were ruled out.
- **disease_name_short**: the disease name only (no agent, no method).
"""

def process_single_file(client, input_path, output_path):
    """
    Reads one JSON file, sends its content to the Gemini API,
    and saves the structured JSON response.
    """
    try:
        # read the source JSON file
        with open(input_path, 'r', encoding='utf-8') as f:
            source_data = json.load(f)
        # convert the source JSON object back into a string to feed to the model
        case_data_string = json.dumps(source_data, indent=2)
        # combine SYSTEM_PROMPT with case data
        full_query = f"{SYSTEM_PROMPT}\n\n---\n\nExtract information from the following clinical case:\n{case_data_string}"

    except json.JSONDecodeError:
        print(f"   ... ERROR: Failed to decode source JSON. Skipping.")
        return
    except Exception as e:
        print(f"   ... ERROR reading file: {e}. Skipping.")
        return

    try:
        config_dict = {
            "response_mime_type": "application/json"
        }

        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=[full_query],
            config=config_dict
        )

        # the API is already forced to return JSON, but clean it just in case
        # just in case it includes markdown backticks
        cleaned_response_text = response.text.strip().replace("```json", "").replace("```", "")

        try:
            output_json = json.loads(cleaned_response_text)
        except json.JSONDecodeError:
            print(f"   ... ERROR: API did not return valid JSON. Response was:")
            print(cleaned_response_text)
            return

        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(output_json, f, indent=2)

        print(f"   ... Success, Saved to {output_path.name}")

    except Exception as e:
        print(f"   ... ERROR during API call: {e}")

def main():
    # batch processor
    print(f"Input folder:  {INPUT_DIR}")
    print(f"Output folder: {OUTPUT_DIR}")
    print(f"Model:         {MODEL_NAME}")
    print("-" * 40)

    OUTPUT_DIR.mkdir(exist_ok=True)
    client = genai.Client(api_key=API_KEY)

    # grab all .json files from the input directory
    input_files = list(INPUT_DIR.glob("*.json"))

    if not input_files:
        print(f"No .json files found in {INPUT_DIR}. Exiting.")
        return

    total_files = len(input_files)
    print(f"Found {total_files} JSON files to process.")

    for i, input_path in enumerate(input_files):
        print(f"\n[{i+1}/{total_files}] Processing: {input_path.name}")
        output_path = OUTPUT_DIR / input_path.name

        # check if output already exists
        if output_path.exists():
            print("   ... Output file already exists. Skipping.")
            continue

        # process the file
        process_single_file(client, input_path, output_path)

        # rate limit
        if i < total_files - 1:
            time.sleep(REQUEST_DELAY_SEC)

    print("\n" + "-" * 40)
    print("Batch processing complete.")

if __name__ == "__main__":
    main()


Input folder:  /content/text_output
Output folder: /content/text_output_chunked
Model:         gemini-2.5-pro
----------------------------------------
Found 93 JSON files to process.

[1/93] Processing: 54---A-52-Year-Old-Male-Safari-Tourist-Returning-fro_2022_Clinical-Cases-in-.json
   ... Output file already exists. Skipping.

[2/93] Processing: 28---A-67-Year-Old-Female-Expatriate-Living-in-Came_2022_Clinical-Cases-in-T.json
   ... Output file already exists. Skipping.

[3/93] Processing: 37---A-29-Year-Old-Woman-from-Malawi-With-Confusi_2022_Clinical-Cases-in-Tro.json
   ... Output file already exists. Skipping.

[4/93] Processing: 91---A-20-Year-Old-Male-from-India-With-Fever-_2022_Clinical-Cases-in-Tropic.json
   ... Output file already exists. Skipping.

[5/93] Processing: 74---A-28-Year-Old-Woman-from-Sierra-Leone-With-_2022_Clinical-Cases-in-Trop.json
   ... Output file already exists. Skipping.

[6/93] Processing: 42---A-41-Year-Old-Male-Traveller-Returning-from-Au_2022_Clini

Text cleanup

In [ ]:
import re
from pathlib import Path

OUTPUT_DIR = Path("/content/knowledge_text_output_chunked")

UNICODE_REPLACEMENTS = {
    r"\u00b0": "°",    # Degree Sign
    r"\u00b2": "²",    # Superscript Two
    r"\u00b5": "µ",    # Micro Sign
    r"\u00ef": "ï",    # Latin Small Letter I with Diaeresis
    r"\u2013": "–",    # En-dash
    r"\u2014": "—",    # Em-dash
    r"\u2019": "’",    # Right Single Quote (Apostrophe)
    r"\u201c": "“",    # Left Double Quotation Mark
    r"\u201d": "”",    # Right Double Quotation Mark
    r"\u2026": "…",    # Ellipsis
    r"\u2079": "⁹",    # Superscript Nine
    r"\u03bc": "μ",    # Greek Small Letter Mu
    r"\u03b2": "β",    # Greek Small Letter Beta
}

def fix_unicode_in_files(target_dir):
    print(f"--- Starting Unicode Fix for JSON files in: {target_dir} ---")

    if not target_dir.exists():
        print(f"Error: Directory not found at {target_dir}")
        return

    json_files = list(target_dir.glob("*.json"))

    if not json_files:
        print("No JSON files found. Exiting.")
        return

    total_files = len(json_files)
    print(f"Found {total_files} JSON files to process.")

    for i, file_path in enumerate(json_files):
        print(f"[{i+1}/{total_files}] Processing: {file_path.name}")
        try:
            content = file_path.read_text(encoding='utf-8')
            modified_content = content
            for escaped_str, replacement_char in UNICODE_REPLACEMENTS.items():
                modified_content = modified_content.replace(escaped_str, replacement_char)
            file_path.write_text(modified_content, encoding='utf-8')
            print("   ... Fixed and saved.")
        except Exception as e:
            print(f"   ... ERROR processing {file_path.name}: {e}")

    print("\n--- Unicode fixing complete. ---")

if __name__ == "__main__":
    fix_unicode_in_files(OUTPUT_DIR)

In [ ]:
import re
from pathlib import Path

OUTPUT_DIR = Path("/content/text_output_chunked")

UNICODE_REPLACEMENTS = {
    r"\u00b0": "°",    # Degree Sign
    r"\u00b2": "²",    # Superscript Two
    r"\u00b5": "µ",    # Micro Sign
    r"\u00ef": "ï",    # Latin Small Letter I with Diaeresis
    r"\u2013": "–",    # En-dash
    r"\u2014": "—",    # Em-dash
    r"\u2019": "’",    # Right Single Quote (Apostrophe)
    r"\u201c": "“",    # Left Double Quotation Mark
    r"\u201d": "”",    # Right Double Quotation Mark
    r"\u2026": "…",    # Ellipsis
    r"\u2079": "⁹",    # Superscript Nine
    r"\u03bc": "μ",    # Greek Small Letter Mu
    r"\u03b2": "β",    # Greek Small Letter Beta
}

def fix_unicode_in_files(target_dir):
    print(f"--- Starting Unicode Fix for JSON files in: {target_dir} ---")

    if not target_dir.exists():
        print(f"Error: Directory not found at {target_dir}")
        return

    json_files = list(target_dir.glob("*.json"))

    if not json_files:
        print("No JSON files found. Exiting.")
        return

    total_files = len(json_files)
    print(f"Found {total_files} JSON files to process.")

    for i, file_path in enumerate(json_files):
        print(f"[{i+1}/{total_files}] Processing: {file_path.name}")
        try:
            content = file_path.read_text(encoding='utf-8')
            modified_content = content
            for escaped_str, replacement_char in UNICODE_REPLACEMENTS.items():
                modified_content = modified_content.replace(escaped_str, replacement_char)
            file_path.write_text(modified_content, encoding='utf-8')
            print("   ... Fixed and saved.")
        except Exception as e:
            print(f"   ... ERROR processing {file_path.name}: {e}")

    print("\n--- Unicode fixing complete. ---")

if __name__ == "__main__":
    fix_unicode_in_files(OUTPUT_DIR)

--- Starting Unicode Fix for JSON files in: /content/text_output_chunked ---
Found 93 JSON files to process.
[1/93] Processing: 54---A-52-Year-Old-Male-Safari-Tourist-Returning-fro_2022_Clinical-Cases-in-.json
   ... Fixed and saved.
[2/93] Processing: 28---A-67-Year-Old-Female-Expatriate-Living-in-Came_2022_Clinical-Cases-in-T.json
   ... Fixed and saved.
[3/93] Processing: 37---A-29-Year-Old-Woman-from-Malawi-With-Confusi_2022_Clinical-Cases-in-Tro.json
   ... Fixed and saved.
[4/93] Processing: 91---A-20-Year-Old-Male-from-India-With-Fever-_2022_Clinical-Cases-in-Tropic.json
   ... Fixed and saved.
[5/93] Processing: 74---A-28-Year-Old-Woman-from-Sierra-Leone-With-_2022_Clinical-Cases-in-Trop.json
   ... Fixed and saved.
[6/93] Processing: 42---A-41-Year-Old-Male-Traveller-Returning-from-Au_2022_Clinical-Cases-in-T.json
   ... Fixed and saved.
[7/93] Processing: 32---A-44-Year-Old-Male-Farmer-from-Laos-With-Di_2022_Clinical-Cases-in-Trop.json
   ... Fixed and saved.
[8/93] Processin

Embedding and upserting

"Chunk"ing and upserting happens here

In [ ]:
import os
import json
import uuid
from pathlib import Path
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer
from config import HUGGINGFACETOKEN, QDRANT_ENDPOINT_URL, QDRANT_API_KEY, logger
from typing import Dict, Any, List
import fitz
import os
import re
import time
from pathlib import Path
from google import genai

Main

In [ ]:
METADATA_DIR = Path("/content/text_output_chunked")
RAW_TEXT_DIR = Path("/content/text_output")

COLLECTION_NAME = "main-rag"
EMBEDDING_MODEL_NAME = 'pritamdeka/S-BioBert-snli-multinli-stsb'
EMBEDDING_DIMENSION = 768
MIN_CHUNK_LENGTH = 50

def get_embedding_model(model_name: str, token: str) -> SentenceTransformer:
    logger.info(f"Loading embedding model: {model_name}")
    try:
        model = SentenceTransformer(model_name, use_auth_token=token)
        return model
    except Exception as e:
        logger.error(f"Failed to load model {model_name}: {e}")
        raise

def get_qdrant_client(url: str, api_key: str) -> QdrantClient:
    if not url or not api_key:
        raise ValueError("Qdrant URL and API Key must be set.")
    logger.info(f"Connecting to Qdrant Cloud at {url}")
    try:
        client = QdrantClient(url=url, api_key=api_key)
        logger.info("Qdrant Cloud connection successful.")
        return client
    except Exception as e:
        logger.error(f"Failed to connect to Qdrant Cloud: {e}")
        raise

def create_qdrant_collection(client: QdrantClient, collection_name: str, embedding_dim: int):
    try:
        client.recreate_collection(
            collection_name=collection_name,
            vectors_config=models.VectorParams(
                size=embedding_dim,
                distance=models.Distance.COSINE
            )
        )
        logger.info(f"Collection '{collection_name}' created/recreated successfully.")

        logger.info("Creating payload index for 'disease_name_short'...")
        client.create_payload_index(
            collection_name=collection_name,
            field_name="disease_name_short",
            field_schema=models.PayloadSchemaType.KEYWORD
        )
        logger.info("Payload index created successfully.")

    except Exception as e:
        logger.error(f"Failed to create collection or index: {e}")
        raise

def main():
    logger.info("Starting Hybrid Embedding and Indexing Process")

    try:
        model = get_embedding_model(EMBEDDING_MODEL_NAME, HUGGINGFACETOKEN)
        client = get_qdrant_client(QDRANT_ENDPOINT_URL, QDRANT_API_KEY)
    except Exception:
        logger.error("Failed to initialize models or clients. Exiting.")
        return

    create_qdrant_collection(client, COLLECTION_NAME, EMBEDDING_DIMENSION)

    metadata_files = list(METADATA_DIR.glob("*.json"))
    if not metadata_files:
        logger.warning(f"No structured .json files found in {METADATA_DIR}. Exiting.")
        return

    logger.info(f"Found {len(metadata_files)} structured JSON files to process.")

    points_batch = []
    batch_size = 32

    for metadata_file_path in metadata_files:
        try:
            with open(metadata_file_path, 'r', encoding='utf-8') as f:
                document_payload = json.load(f)

            raw_text_file_path = RAW_TEXT_DIR / metadata_file_path.name
            if not raw_text_file_path.exists():
                logger.warning(f"No matching raw text file at {raw_text_file_path} for {metadata_file_path.name}. Skipping.")
                continue

            with open(raw_text_file_path, 'r', encoding='utf-8') as f:
                raw_data = json.load(f)

            page_texts = raw_data.get("pages", [])
            if not page_texts:
                logger.warning(f"No 'pages' found in {raw_text_file_path.name}. Skipping.")
                continue

            for page_num, page_item in enumerate(page_texts):
                chunk_text = ""
                try:
                    if isinstance(page_item, str):
                        chunk_text = page_item.strip()
                    elif isinstance(page_item, dict):
                        if "text" in page_item and isinstance(page_item["text"], str):
                            chunk_text = page_item["text"].strip()
                        elif "content" in page_item and isinstance(page_item["content"], str):
                            chunk_text = page_item["content"].strip()
                        elif "page_content" in page_item and isinstance(page_item["page_content"], str):
                            chunk_text = page_item["page_content"].strip()
                        else:
                            chunk_text = json.dumps(page_item)
                    elif page_item is not None:
                        chunk_text = str(page_item).strip()
                except Exception as e:
                    continue

                if len(chunk_text) < MIN_CHUNK_LENGTH:
                    continue

                vector = model.encode(chunk_text).tolist()

                point_payload = document_payload.copy()
                point_payload['raw_text_chunk'] = chunk_text
                point_payload['source_filename'] = metadata_file_path.name
                point_payload['page_number'] = page_num + 1

                point = models.PointStruct(
                    id=str(uuid.uuid4()),
                    vector=vector,
                    payload=point_payload
                )
                points_batch.append(point)

                if len(points_batch) >= batch_size:
                    client.upsert(collection_name=COLLECTION_NAME, points=points_batch, wait=True)
                    logger.info(f"Upserted batch of {len(points_batch)} points.")
                    points_batch = []

        except Exception as e:
            logger.error(f"Error processing file {metadata_file_path.name}: {e}")

    if points_batch:
        client.upsert(collection_name=COLLECTION_NAME, points=points_batch, wait=True)
        logger.info(f"Upserted final batch of {len(points_batch)} points.")

    logger.info("Hybrid Embedding and Indexing Process Complete")

if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/sentence_transformers/SentenceTransformer.py:204: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v4 of SentenceTransformers.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipython-input-3383420353.py:32: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


Knowledge

In [ ]:
METADATA_DIR = Path("/content/knowledge_text_output_chunked")
RAW_TEXT_DIR = Path("/content/knowledge_text_output")

COLLECTION_NAME = "knowledge-rag"
EMBEDDING_MODEL_NAME = 'pritamdeka/S-BioBert-snli-multinli-stsb'
EMBEDDING_DIMENSION = 768
MIN_CHUNK_LENGTH = 50

def get_embedding_model(model_name: str, token: str) -> SentenceTransformer:
    logger.info(f"Loading embedding model: {model_name}")
    try:
        model = SentenceTransformer(model_name, use_auth_token=token)
        return model
    except Exception as e:
        logger.error(f"Failed to load model {model_name}: {e}")
        raise

def get_qdrant_client(url: str, api_key: str) -> QdrantClient:
    if not url or not api_key:
        raise ValueError("Qdrant URL and API Key must be set.")
    logger.info(f"Connecting to Qdrant Cloud at {url}")
    try:
        client = QdrantClient(url=url, api_key=api_key)
        logger.info("Qdrant Cloud connection successful.")
        return client
    except Exception as e:
        logger.error(f"Failed to connect to Qdrant Cloud: {e}")
        raise

def create_qdrant_collection(client: QdrantClient, collection_name: str, embedding_dim: int):
    try:
        client.recreate_collection(
            collection_name=collection_name,
            vectors_config=models.VectorParams(
                size=embedding_dim,
                distance=models.Distance.COSINE
            )
        )
        logger.info(f"Collection '{collection_name}' created/recreated successfully.")

        logger.info("Creating payload index for 'disease_name_short'...")
        client.create_payload_index(
            collection_name=collection_name,
            field_name="disease_name_short",
            field_schema=models.PayloadSchemaType.KEYWORD
        )
        logger.info("Payload index created successfully.")

    except Exception as e:
        logger.error(f"Failed to create collection or index: {e}")
        raise

def main():
    logger.info("Starting Hybrid Embedding and Indexing Process")

    try:
        model = get_embedding_model(EMBEDDING_MODEL_NAME, HUGGINGFACETOKEN)
        client = get_qdrant_client(QDRANT_ENDPOINT_URL, QDRANT_API_KEY)
    except Exception:
        logger.error("Failed to initialize models or clients. Exiting.")
        return

    create_qdrant_collection(client, COLLECTION_NAME, EMBEDDING_DIMENSION)

    metadata_files = list(METADATA_DIR.glob("*.json"))
    if not metadata_files:
        logger.warning(f"No structured .json files found in {METADATA_DIR}. Exiting.")
        return

    logger.info(f"Found {len(metadata_files)} structured JSON files to process.")

    points_batch = []
    batch_size = 32

    for metadata_file_path in metadata_files:
        try:
            with open(metadata_file_path, 'r', encoding='utf-8') as f:
                document_payload = json.load(f)

            raw_text_file_path = RAW_TEXT_DIR / metadata_file_path.name
            if not raw_text_file_path.exists():
                logger.warning(f"No matching raw text file at {raw_text_file_path} for {metadata_file_path.name}. Skipping.")
                continue

            with open(raw_text_file_path, 'r', encoding='utf-8') as f:
                raw_data = json.load(f)

            page_texts = raw_data.get("pages", [])
            if not page_texts:
                logger.warning(f"No 'pages' found in {raw_text_file_path.name}. Skipping.")
                continue

            for page_num, page_item in enumerate(page_texts):
                chunk_text = ""
                try:
                    if isinstance(page_item, str):
                        chunk_text = page_item.strip()
                    elif isinstance(page_item, dict):
                        if "text" in page_item and isinstance(page_item["text"], str):
                            chunk_text = page_item["text"].strip()
                        elif "content" in page_item and isinstance(page_item["content"], str):
                            chunk_text = page_item["content"].strip()
                        elif "page_content" in page_item and isinstance(page_item["page_content"], str):
                            chunk_text = page_item["page_content"].strip()
                        else:
                            chunk_text = json.dumps(page_item)
                    elif page_item is not None:
                        chunk_text = str(page_item).strip()
                except Exception as e:
                    continue

                if len(chunk_text) < MIN_CHUNK_LENGTH:
                    continue

                vector = model.encode(chunk_text).tolist()

                point_payload = document_payload.copy()
                point_payload['raw_text_chunk'] = chunk_text
                point_payload['source_filename'] = metadata_file_path.name
                point_payload['page_number'] = page_num + 1

                point = models.PointStruct(
                    id=str(uuid.uuid4()),
                    vector=vector,
                    payload=point_payload
                )
                points_batch.append(point)

                if len(points_batch) >= batch_size:
                    client.upsert(collection_name=COLLECTION_NAME, points=points_batch, wait=True)
                    logger.info(f"Upserted batch of {len(points_batch)} points.")
                    points_batch = []

        except Exception as e:
            logger.error(f"Error processing file {metadata_file_path.name}: {e}")

    if points_batch:
        client.upsert(collection_name=COLLECTION_NAME, points=points_batch, wait=True)
        logger.info(f"Upserted final batch of {len(points_batch)} points.")

    logger.info("Hybrid Embedding and Indexing Process Complete")

if __name__ == "__main__":
    main()

/tmp/ipython-input-821926825.py:32: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


THE RAG IMPLEMENTATION

IMPORANT PLEASE RUN THIS BEFORE EVERYTHING ELSE QUERY RELATED AND DONT CHANGE ANYTHING UNLESS COSMETIC

In [ ]:
import os
from google import genai
import logging
import sys
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer
import json
from config import QDRANT_ENDPOINT_URL, QDRANT_API_KEY, HUGGINGFACETOKEN, GEMINI_API_KEY, logger
from typing import List, Dict, Any, Optional
from abc import ABC, abstractmethod

try:
    API_KEY = os.environ["GEMINI_API_KEY"]
except KeyError:
    print("!! ERROR: GEMINI_API_KEY not set.")
    raise

try:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    LOCAL_LIBS_AVAILABLE = True
except ImportError:
    LOCAL_LIBS_AVAILABLE = False

# configuration
EMBEDDING_MODEL_NAME = 'pritamdeka/S-BioBert-snli-multinli-stsb'
COLLECTION_NAME = "main-rag"
GENERATOR_MODEL_NAME = 'google/medgemma-4b-it' #change to gemini-2.5-pro OR google/medgemma-4b-it. Running finetuned model has its own function below
FILTER_MODEL_NAME = 'gemini-2.5-flash'


class GeneratorStrategy(ABC):
    def __init__(self, model_name: str):
        self.model_name = model_name
    @abstractmethod
    def generate(self, prompt: str) -> str:
        pass

class GeminiAPIStrategy(GeneratorStrategy):
    def __init__(self, model_name: str, client: genai.Client, is_json_output: bool = False):
        super().__init__(model_name)
        self.client = client
        self.is_json_output = is_json_output

    def generate(self, prompt: str) -> str:
        safety_settings_list = [
            {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"}
        ]
        config = {"safety_settings": safety_settings_list}
        if self.is_json_output:
            config["response_mime_type"] = "application/json"
        try:
            response = self.client.models.generate_content(
                model=self.model_name, contents=[prompt], config=config
            )
            return response.text
        except Exception as e:
            logger.error(f"Error during Gemini API call: {e}")
            return f"Error generating response: {e}"

class LocalHuggingFaceStrategy(GeneratorStrategy):
    def __init__(self, model_name: str):
        super().__init__(model_name)
        if not LOCAL_LIBS_AVAILABLE:
            raise ImportError("LocalHuggingFaceStrategy requires 'transformers' and 'torch'.")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self._load_model()

    def _load_model(self):
        logger.info(f"[Strategy] Loading local model: {self.model_name}...")

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name,
            token=os.environ.get("HF_TOKEN")
        )
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            token=os.environ.get("HF_TOKEN")
        )
        logger.info(f"[Strategy] Local model {self.model_name} loaded successfully.")

    def generate(self, prompt: str) -> str:
        chat_prompt = [{"role": "user", "content": prompt}]
        inputs = self.tokenizer.apply_chat_template(
            chat_prompt, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to(self.device)

        if inputs.shape[1] > 2000:
             inputs = inputs[:, -2000:]

        output_ids = self.model.generate(
            inputs,
            max_new_tokens=1500,
            do_sample=True,
            temperature=0.7,
            top_p=0.95
        )
        response_text = self.tokenizer.batch_decode(
            output_ids[:, inputs.shape[1]:], skip_special_tokens=True
        )[0]
        return response_text.strip()

class GeneratorFactory:
    def __init__(self, genai_client: genai.Client):
        self.genai_client = genai_client
        self.loaded_local_models = {}

    def create_generator(self, model_name: str) -> GeneratorStrategy:
        if model_name.startswith("gemini-"):
            is_json = "json" in model_name
            return GeminiAPIStrategy(model_name, self.genai_client, is_json_output=is_json)
        elif model_name.startswith("google/medgemma") or "/" in model_name:
            if model_name not in self.loaded_local_models:
                self.loaded_local_models[model_name] = LocalHuggingFaceStrategy(model_name)
            return self.loaded_local_models[model_name]
        elif model_name.startswith("fine_tuned") or "fine" in model_name:
            if model_name not in self.loaded_local_models:
                self.loaded_local_models[model_name] = LocalHuggingFaceStrategy(model_name)
            return self.loaded_local_models[model_name]
        else:
            raise ValueError(f"Unknown model type or name: {model_name}")

class RAGAgent:
    def __init__(self):
        logger.info("Initializing Dual-Source RAGAgent")

        self.EMBEDDING_MODEL_NAME = EMBEDDING_MODEL_NAME
        self.MAIN_COLLECTION = COLLECTION_NAME       # "main-rag" (Source of Truth)
        self.KNOWLEDGE_COLLECTION = "knowledge-rag"  # knowledge rag (Book/Background)
        self.GENERATOR_MODEL_NAME = GENERATOR_MODEL_NAME

        logger.info(f"Loading embedding model: {self.EMBEDDING_MODEL_NAME} on CPU")
        self.embed_model = SentenceTransformer(
            self.EMBEDDING_MODEL_NAME,
            use_auth_token=HUGGINGFACETOKEN,
            device="cpu"
        )

        logger.info(f"Connecting to Qdrant Cloud...")
        self.qdrant_client = QdrantClient(
            url=QDRANT_ENDPOINT_URL,
            api_key=QDRANT_API_KEY
        )
        #test testtesttesttesttesttesttesttesttesttesttesttesttesttesttesttesttesttesttesttesttesttesttesttesttesttesttesttesttesttesttesttesttesttesttesttesttesttesttes
        self.genai_client = genai.Client(api_key=GEMINI_API_KEY)
        self.generator_factory = GeneratorFactory(self.genai_client)

        self.filter_generator = GeminiAPIStrategy(
            model_name=FILTER_MODEL_NAME,
            client=self.genai_client,
            is_json_output=True
        )

        self.answer_generator = self.generator_factory.create_generator(self.GENERATOR_MODEL_NAME)
        logger.info("RAGAgent initialized successfully.")

    def extract_filters_from_query(self, query: str) -> Optional[models.Filter]:
        logger.info("Extracting filters from query...")
        prompt = f"""
        You are a query analysis assistant. Extract *only* the name of a disease.
        Respond with *only* a JSON object: {{"disease_name_short": "Name of Disease"}}
        If no disease is mentioned, return: {{"disease_name_short": null}}
        User Query: "{query}"
        """
        try:
            response_text = self.filter_generator.generate(prompt)
            cleaned_text = response_text.replace("```json", "").replace("```", "").strip()
            filter_data = json.loads(cleaned_text)
            disease_name = filter_data.get("disease_name_short")

            if disease_name:
                if " " in disease_name:
                    disease_name = disease_name.split()[0]
                logger.info(f"Found filter: {disease_name}")
                return models.Filter(must=[
                    models.FieldCondition(
                        key="disease_name_short",
                        match=models.MatchText(text=disease_name)
                    )
                ])
            return None
        except Exception as e:
            logger.error(f"Filter extraction error: {e}")
            return None

    def search_knowledge_base(self, query: str, filters: models.Filter = None, top_k: int = 5) -> List[Dict[str, Any]]:
        logger.info(f"Embedding query...")
        query_vector = self.embed_model.encode(query).tolist()

        results = []

        logger.info(f"Searching MAIN RAG (Collection: {self.MAIN_COLLECTION})...")
        main_results = self.qdrant_client.query_points(
            collection_name=self.MAIN_COLLECTION,
            query=query_vector,
            query_filter=filters,
            limit=top_k,
            with_payload=True
        )

        for point in main_results.points:
            payload = point.payload
            payload['source_tier'] = 'OFFICIAL_SOURCE' # make main high prio
            results.append(payload)

        logger.info(f"Searching KNOWLEDGE RAG (Collection: {self.KNOWLEDGE_COLLECTION})...")
        try:
            knowledge_results = self.qdrant_client.query_points(
                collection_name=self.KNOWLEDGE_COLLECTION,
                query=query_vector,
                limit=3,
                with_payload=True
            )
            for point in knowledge_results.points:
                payload = point.payload
                if 'page_content' in payload and 'raw_text_chunk' not in payload:
                    payload['raw_text_chunk'] = payload['page_content']

                payload['source_tier'] = 'BACKGROUND_INFO' # maek the other one low prio
                results.append(payload)
        except Exception as e:
            logger.warning(f"Knowledge search skipped (collection might not exist yet): {e}")

        logger.info(f"Retrieved {len(results)} total contexts from both collections.")
        return results

    def build_prompt(self, query: str, context_docs: List[Dict[str, Any]]) -> str:
        context_docs.sort(key=lambda x: x.get('source_tier', 'z'), reverse=True)

        context_str_list = []
        for doc in context_docs:
            tier = doc.get('source_tier', 'UNKNOWN')

            if tier == 'OFFICIAL_SOURCE':
                snippet = (
                    f"=== OFFICIAL MEDICARE DATA (SOURCE OF TRUTH) ===\n"
                    f"Source File: {doc.get('source_filename', 'N/A')}\n"
                    f"Case Diagnosis: {doc.get('disease_name_short', 'N/A')}\n"
                    f"Detailed Diagnosis: {doc.get('final_diagnosis', 'N/A')}\n"
                    f"Chief Complaints: {doc.get('chief_complaint', 'N/A')}\n"
                    f"History of Present Illnesses: {doc.get('history_of_present_illness', 'N/A')}\n"
                    f"Physical Examinations: {doc.get('physical_exam', 'N/A')}\n"
                    f"Lab Diagnostics Results: {doc.get('labs_and_diagnostics', 'N/A')}\n"
                    f"Differential Diagnosis: {doc.get('differential_diagnosis', 'N/A')}\n"
                    f"Relevant Text Snippet: {doc.get('raw_text_chunk', 'N/A')}\n"
                )
            else:
                snippet = (
                    f"=== BACKGROUND/BOOK INFO (SECONDARY CONTEXT) ===\n"
                    f"Source: {doc.get('source_filename', 'General Knowledge')}\n"
                    f"Content: {doc.get('raw_text_chunk', 'N/A')}\n"
                )
            context_str_list.append(snippet)

        context_str = "\n\n---\n\n".join(context_str_list)

        prompt = f"""You are an expert clinical diagnostic assistant.

        **INSTRUCTIONS ON SOURCES:**
        1. **OFFICIAL MEDICARE DATA** is your absolute Source of Truth. Prioritize this above all else.
        2. **BACKGROUND/BOOK INFO** is secondary. Do NOT use schemas, templates, or formats from this source. Only use it to explain concepts if the Official Data is unclear.
        3. If the sources conflict, IGNORE the Background Info.
        4. For every diagnostic query, open a short and concise answer on the suspected disease or illness the patient has. Then, elborate for 3 to 6 sentence on how you came to the conclusion Based on info that was given.
        5. Absolutely avoid using any other information that isn't included in the Official Medicare Data or Background/Book Info.

        **PROVIDED CONTEXT:**
        {context_str}

        **USER QUESTION:**
        {query}

        **ASSISTANT ANSWER:**
        """
        return prompt

    def ask(self, query: str) -> tuple[str, list[str]]:
        qdrant_filter = self.extract_filters_from_query(query)
        retrieved_contexts = self.search_knowledge_base(query, filters=qdrant_filter)

        if not retrieved_contexts and qdrant_filter is not None:
            logger.warning(f"Hybrid search failed (0 results). Retrying pure vector search.")
            retrieved_contexts = self.search_knowledge_base(query, filters=None)

        context_strings = [doc.get('raw_text_chunk', '') for doc in retrieved_contexts]

        if not retrieved_contexts:
            return "I'm sorry, I could not find any relevant information.", []

        prompt = self.build_prompt(query, retrieved_contexts)

        logger.info(f"Generating answer using {self.answer_generator.model_name}...")
        try:
            answer_text = self.answer_generator.generate(prompt)
            return answer_text, context_strings
        except Exception as e:
            logger.error(f"Error: {e}")
            return f"Error: {e}", context_strings

In [ ]:
from qdrant_client import QdrantClient, models
from config import QDRANT_ENDPOINT_URL, QDRANT_API_KEY


client = QdrantClient(url=QDRANT_ENDPOINT_URL, api_key=QDRANT_API_KEY)
COLLECTION_NAME = "main-rag"

print(f"Creating TEXT index for 'disease_name_short' in {COLLECTION_NAME}...")

client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="disease_name_short",
    field_schema=models.PayloadSchemaType.TEXT
)

print("SUCCESS: Text Index created. You can now run the RAG query.")

Creating TEXT index for 'disease_name_short' in main-rag...
SUCCESS: Text Index created. You can now run the RAG query.


QUERY NORMAL (SWITCH MODELS ABOVE)


In [ ]:
#load the model
agent = RAGAgent()

# query
query = "An NS1 test is performed on a patient, and detects high concentrations of NS1 protein within their blood. A Reverse Transcriptase PCR (RT-PCR) test also reveals dengue virus RNA in the blood. What is this patient's initial diagnosis?"
print(f"\nQuery: {query}")
answer, contexts = agent.ask(query)

print("\n--- GENERATED ANSWER ---")
print(answer)
print("------------------------")

/usr/local/lib/python3.12/dist-packages/sentence_transformers/SentenceTransformer.py:204: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v4 of SentenceTransformers.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


Query: An NS1 test is performed on a patient, and detects high concentrations of NS1 protein within their blood. A Reverse Transcriptase PCR (RT-PCR) test also reveals dengue virus RNA in the blood. What is this patient's initial diagnosis?

--- GENERATED ANSWER ---
The patient's initial diagnosis is likely **Dengue Virus Infection**.

Here's the reasoning:

*   **NS1 Protein Detection:** The presence of high concentrations of NS1 (Non-Structural Protein 1) in the blood is a highly specific marker for Dengue Virus infection. NS1 is a protein produced by the Dengue Virus and is released into the bloodstream.

*   **RT-PCR Confirmation:** The RT-PCR test confirms the presence of Dengue Virus RNA in the blood, further supporting the diagnosis. RT-PCR is a highly sensitive test that detects the viral genome.

*   **Clinical Context:** While NS1 and RT-PCR are key diagnostic tools, it's important to consider the patient's clinical presentation (fever, rash, body aches, etc.) to confirm the

Fine-tuned model query

In [ ]:
from unsloth import FastLanguageModel
import torch


if 'model' not in globals():
    print("Loading 'lora_model'...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model",
        max_seq_length = 2048,
        dtype = None,
        load_in_4bit = True,
    )
    FastLanguageModel.for_inference(model)


class RobustGemmaStrategy(GeneratorStrategy):
    def __init__(self, model, tokenizer):
        super().__init__("gemma3_robust")
        self.model = model
        self.tokenizer = tokenizer
        self.device = "cuda"

    def generate(self, prompt: str) -> str:
        formatted_prompt = f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"

        try:
            inputs = self.tokenizer(text=formatted_prompt, return_tensors="pt")

            if inputs is None:
                return "Error: Tokenizer returned None."


            if isinstance(inputs, dict) or hasattr(inputs, "keys"):
                input_ids = inputs["input_ids"]
                input_ids = input_ids.to(self.device)
            else:
                return f"Error: Unexpected tokenizer output type: {type(inputs)}"

            outputs = self.model.generate(
                input_ids,
                max_new_tokens=500,
                do_sample=True,
                temperature=0.7,
                top_p=0.95,
                pad_token_id=self.tokenizer.pad_token_id
            )


            generated_ids = outputs[0][input_ids.shape[1]:]
            response = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
            return response.strip()

        except Exception as e:
            return f"Generation Error: {e}"


temp_name = GENERATOR_MODEL_NAME
GENERATOR_MODEL_NAME = "google/medgemma-4b-it"
agent = RAGAgent()
GENERATOR_MODEL_NAME = temp_name

print("Swapping to Fine Tuning MedGemma...")
agent.answer_generator = RobustGemmaStrategy(model, tokenizer)

query = "An NS1 test is performed on a patient, and detects high concentrations of NS1 protein within their blood. A Reverse Transcriptase PCR (RT-PCR) test also reveals dengue virus RNA in the blood. What is this patient's initial diagnosis?"
print(f"Query: {query}")

answer, contexts = agent.ask(query)

print("\n--- GENERATED ANSWER ---")
print(answer)
print("------------------------")

/usr/local/lib/python3.12/dist-packages/sentence_transformers/SentenceTransformer.py:204: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v4 of SentenceTransformers.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Swapping to Robust Strategy...
Query: An NS1 test is performed on a patient, and detects high concentrations of NS1 protein within their blood. A Reverse Transcriptase PCR (RT-PCR) test also reveals dengue virus RNA in the blood. What is this patient's initial diagnosis?

--- GENERATED ANSWER ---
Diagnosis: Severe Dengue

Clinical Reasoning: The final diagnosis was severe dengue. This was based on the typical clinical progression from a febrile phase to a critical phase at the time of defervescence, characterized by signs of vascular leakage (pleural effusions, ascites, haemoconcentration) and compensated shock. The diagnosis was definitively confirmed by a positive dengue NS1 ELISA and a positive RT-PCR for dengue virus.
------------------------




---



EVALUATION


In [ ]:
# eval set file creation
eval_set_content = r"""
EVAL_SET = [
    {
        "question": "A 32-year-old patient presents to the emergency department with high fever, shaking chills, and profuse sweating that occur every 48 hours. They report severe headaches, body aches, and nausea. The patient returned two weeks ago from a month-long trip to sub-Saharan Africa. A physical exam reveals a slightly enlarged spleen and mild jaundice. The physician orders a blood test, specifically requesting a thick and thin blood smear. The lab results confirm the presence of parasites infecting the red blood cells. The patient is immediately started on antimalarial medication.",
        "ground_truth": "Malaria. The patient's symptoms of fever, chills, and headache following travel to an endemic area like sub-Saharan Africa are classic indicators of malaria. The \"gold standard\" for diagnosis, as described in the provided CDC text, is the microscopic examination of thick and thin blood smears to detect the parasite. The presence of parasites in the red blood cells confirms the diagnosis."
    },
    {
        "question": "A young man from rural South America seeks medical attention for a sudden, painless swelling of his right eyelid. He also reports a low-grade fever, fatigue, and body aches. He lives in a house with a thatched roof and adobe walls. The doctor suspects a vector-borne disease transmitted by an insect often found in the crevices of such walls. Years later, if left untreated, the patient is at risk for serious cardiac issues like irregular heartbeats or heart failure. The physician considers the possibility of infection by *Trypanosoma cruzi*.",
        "ground_truth": "Chagas disease. The painless swelling of the eyelid is a classic sign (often called Romaña's sign) of acute Chagas disease, caused by the *Trypanosoma cruzi* parasite. The patient's living conditions (adobe walls, thatched roof) increase exposure to the triatomine bug (kissing bug) that transmits the disease. The Mayo Clinic link notes that chronic complications can include serious heart problems like heart failure and digestive issues."
    },
    {
        "question": "A soldier returning from the Middle East notices several skin sores on his arms that have developed over the past few months. The sores started as small bumps but have enlarged into ulcerated lesions with raised edges, resembling a volcano with a central crater. They are largely painless but have not healed. He recalls being bitten by small, silent flying insects during his deployment. A tissue sample from the sores is taken for microscopic examination. The medical team is evaluating him for a parasitic infection transmitted by sand flies.",
        "ground_truth": "Leishmaniasis (Cutaneous). The description of skin sores with a \"volcano-like\" appearance (raised edges and central crater) following exposure to sand flies is characteristic of Cutaneous Leishmaniasis. The CDC information highlights that these sores can be painless and may take months or years to heal. Diagnosis is typically confirmed by examining tissue samples to detect the parasite."
    },
    {
        "question": "A 12-year-old child from West Africa presents with a large, painless swelling on their left leg. Over the course of four weeks, the swelling has progressed to form a large ulcer with a whitish-yellow base. Surprisingly, the child reports very little pain despite the extent of the tissue damage. The doctor explains that the bacteria causing this produces a toxin called mycolactone, which suppresses the immune system and dulls pain. The infection is associated with exposure to stagnant bodies of water. Treatment will involve a combination of antibiotics.",
        "ground_truth": "Buruli ulcer. The case describes the typical presentation of Buruli ulcer, caused by *Mycobacterium ulcerans*: a painless swelling that evolves into a large ulcer. The WHO fact sheet explains that the toxin mycolactone destroys tissue while suppressing pain and the immune response. The geographic context (West Africa) and association with stagnant water further support this diagnosis."
    },
    {
        "question": "A traveler who visited a rural village in Chad a year ago develops a fever and swelling in his lower leg. A few days later, a painful blister forms on his foot, accompanied by a burning sensation. To relieve the pain, he submerges his foot in water, which causes the blister to burst and a white worm to emerge. The doctor identifies this as a parasitic infection acquired by drinking unfiltered stagnant water containing infected copepods. The treatment involves slowly removing the worm over several days or weeks.",
        "ground_truth": "Guinea worm disease (Dracunculiasis). The emergence of a worm from a painful blister on the leg roughly one year after infection is the hallmark of Guinea worm disease. As noted in the CDC text, the infection is contracted by drinking water contaminated with parasite-infected copepods. The burning pain often drives patients to seek relief in water, triggering the release of larvae."
    },
    {
        "question": "A tourist returning from a tropical region presents with sudden onset of fever, chills, severe back pain, and headache. After a brief improvement, the symptoms return more severely, accompanied by abdominal pain and vomiting. The physician observes that the patient's skin and eyes have turned yellow. The patient mentions being bitten by mosquitoes during the trip. The doctor is concerned about the \"toxic phase\" of this viral disease, which can lead to kidney failure and bleeding. Diagnosis requires blood-sample testing with PCR.",
        "ground_truth": "Yellow fever. The symptoms of fever, back pain, and subsequent jaundice (yellowing of the skin) are defining features of Yellow fever. The Wikipedia entry describes the \"toxic phase\" that affects about 15% of patients, characterized by recurring fever, liver damage (jaundice), and bleeding. The transmission by mosquitoes in tropical areas is also a key diagnostic clue."
    },
    {
        "question": "A patient is brought to the hospital exhibiting extreme anxiety, confusion, and agitation. They complain of a prickling and itching sensation at the site of a dog bite that occurred several months ago while traveling. The patient refuses to drink water, showing signs of hydrophobia, and has difficulty swallowing. The medical team suspects a viral infection that targets the central nervous system. Samples of saliva and a skin biopsy from the nape of the neck are collected for laboratory testing to confirm the diagnosis.",
        "ground_truth": "Rabies. The progression from a bite to symptoms like prickling at the wound site, anxiety, and hydrophobia (fear of water) points directly to Rabies. The CDC clinical care page outlines these neurologic signs and specifies that antemortem diagnosis involves testing saliva and nuchal skin biopsies. The disease is nearly always fatal once symptoms appear."
    },
    {
        "question": "An elderly man in Guyana visits a clinic due to severe, permanent swelling of his left leg, which has become thickened and hardened, resembling an elephant's leg. He also reports a history of scrotal swelling. He recalls being bitten frequently by mosquitoes in the past. The condition causes him significant disability and social stigma. The health authorities mention that this disease is caused by microscopic worms lodging in the lymphatic system. Treatment involves managing the swelling and preventing secondary skin infections.",
        "ground_truth": "Lymphatic filariasis. The massive swelling of the leg (elephantiasis) and scrotum (hydrocele) are the visible manifestations of Lymphatic filariasis. The PAHO text explains that the disease is caused by parasitic worms transmitted by mosquitoes, which damage the lymphatic system. The reference to \"Bigfoot\" in Guyana is a specific local term mentioned in the provided literature."
    },
    {
        "question": "A patient returning from a month-long backpacking trip in rural agricultural areas of Asia presents with high fever, severe headache, and vomiting. Upon admission, the patient becomes disoriented and suffers a seizure. The doctors suspect a viral encephalitis transmitted by mosquitoes. They order a lumbar puncture to test the cerebrospinal fluid for IgM antibodies against the virus. There is no specific cure, so treatment focuses on supportive care to manage the severe neurologic symptoms.",
        "ground_truth": "Japanese encephalitis. The travel history to rural Asia, combined with symptoms of encephalitis (fever, headache, seizures), strongly suggests Japanese encephalitis. The CDC highlights that while many infections are mild, a small percentage develop into severe brain inflammation. Diagnosis is typically confirmed by detecting IgM antibodies in the cerebrospinal fluid."
    },
    {
        "question": "A group of travelers returning from a swimming trip in a freshwater lake in Africa develop an itchy rash (swimmer's itch). Weeks later, one traveler experiences fever, cough, muscle aches, and hives. Blood tests reveal a high eosinophil count. The physician suspects an infection caused by parasitic worms that penetrate the skin during contact with contaminated water. Stool and urine samples are collected to look for microscopic eggs. The condition is sometimes referred to as Katayama fever during its acute phase.",
        "ground_truth": "Schistosomiasis. The history of swimming in freshwater in Africa, followed by \"swimmer's itch\" and later acute symptoms like fever and eosinophilia (Katayama fever), indicates Schistosomiasis. The CDC Yellow Book explains that the infection is caused by parasitic worms released by snails, and diagnosis involves finding eggs in stool or urine or detecting antibodies."
    }
]
"""

with open("evaluation_set.py", "w") as f:
    f.write(eval_set_content)

print("Evaluation set created successfully.")

Evaluation set created successfully.


**SOURCES USED**

https://www.cdc.gov/malaria/hcp/diagnosis-testing/malaria-diagnostic-tests.html
https://www.cdc.gov/dengue/media/pdfs/2024/05/20240521_342849-B_PRESS_READY_PocketGuideDCMC_UPDATE.pdf
https://www.mayoclinic.org/diseases-conditions/chagas-disease/symptoms-causes/syc-20356212
https://www.cdc.gov/leishmaniasis/signs-symptoms/index.html
https://www.who.int/news-room/fact-sheets/detail/buruli-ulcer-(mycobacterium-ulcerans-infection)
https://www.cdc.gov/sth/about/index.html
https://docs.bvsalud.org/biblioref/2023/09/1451941/detecting-onchocerca-volvulus-infection-in-low-prevalence-area_kGS6RIe.pdf
https://www.cdc.gov/guinea-worm/about/index.html
https://ph.health.mil/cdt/cphe-cdt-trypanosomiasis-ref.pdf
https://en.wikipedia.org/wiki/Yellow_fever
https://www.cdc.gov/rabies/hcp/clinical-care/index.html
https://pmc.ncbi.nlm.nih.gov/articles/PMC3316859/
https://www.paho.org/en/topics/lymphatic-filariasis
https://www.cdc.gov/japanese-encephalitis/symptoms-diagnosis-treatment/index.html
https://en.wikipedia.org/wiki/Yaws
https://www.who.int/news-room/fact-sheets/detail/zika-virus
https://timesofindia.indiatimes.com/life-style/health-fitness/health-news/new-hope-for-snakebite-victims-scientists-reveal-next-gen-antivenom-that-could-revolutionise-treatment-for-17-african-snake-species/articleshow/125062747.cms
https://archive.cdc.gov/www_cdc_gov/grand-rounds/pp/2015/20150519-pdf-dengue-chikungunya-508.pdf
https://ph.health.mil/cdt/cphe-cdt-leprosy-ref.pdf
https://www.cdc.gov/yellow-book/hcp/travel-associated-infections-diseases/schistosomiasis.html

Running the actual evaluation

In [ ]:
import os
import pandas as pd
import asyncio
import logging
import warnings
import importlib
from google.colab import userdata

try:
    SECURE_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    SECURE_KEY = os.environ.get('GEMINI_API_KEY')

if not SECURE_KEY:
    import config
    SECURE_KEY = getattr(config, 'GEMINI_API_KEY', None)

if not SECURE_KEY:
    raise ValueError("CRITICAL: GEMINI_API_KEY is missing. Check Colab Secrets (Key symbol on the left).")

os.environ["GOOGLE_API_KEY"] = SECURE_KEY
os.environ["GEMINI_API_KEY"] = SECURE_KEY

import config
config.GEMINI_API_KEY = SECURE_KEY
importlib.reload(config)

from evaluation_set import EVAL_SET
from ragas import evaluate
from ragas.run_config import RunConfig
from datasets import Dataset
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from unsloth import FastLanguageModel

warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("google_genai._api_client").setLevel(logging.ERROR)

try:
    from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall, answer_correctness
except ImportError:
    from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall, answer_correctness

logger.info("Re-Initializing RAGAgent with verified API Key...")
agent = RAGAgent()
agent.genai_client = genai.Client(api_key=SECURE_KEY)
agent.generator_factory.genai_client = agent.genai_client
agent.answer_generator = agent.generator_factory.create_generator(agent.GENERATOR_MODEL_NAME)

factory = agent.generator_factory

judge_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-pro",
    temperature=0,
    max_retries=10,
    google_api_key=SECURE_KEY
)

EMBEDDING_MODEL_NAME = 'pritamdeka/S-BioBert-snli-multinli-stsb'
hf_embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={'token': config.HUGGINGFACETOKEN}
)

logger.info("Loading Fine-Tuned MedGemma...")
ft_model_path = "lora_model"
ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name = ft_model_path,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(ft_model)

class LocalMedGemmaWrapper:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.tok = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
        self.model_name = "medgemma_finetuned"

    def generate(self, prompt: str):
        formatted_prompt = f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
        inputs = self.tok(formatted_prompt, return_tensors="pt").to("cuda")
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=300,
            use_cache=True,
            pad_token_id=self.tok.eos_token_id
        )
        return self.tok.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

local_medgemma_generator = LocalMedGemmaWrapper(ft_model, ft_tokenizer)

CONFIGS = [
    {"name": "Gemini_RAG",        "model": "gemini-2.5-pro",       "use_rag": True},
    {"name": "Gemini_Raw",        "model": "gemini-2.5-pro",       "use_rag": False},
    {"name": "MedGemma_RAG",      "model": "google/medgemma-4b-it",      "use_rag": True},
    {"name": "MedGemma_Raw",      "model": "google/medgemma-4b-it",      "use_rag": False},
    {"name": "MedGemma_Finetuned_RAG", "model": "fine_tuned",            "use_rag": True},
    {"name": "MedGemma_Finetuned_Raw", "model": "fine_tuned",            "use_rag": False},
]

all_results = []
logger.info(f"Starting 6-Way Evaluation over {len(EVAL_SET)} questions...")

for i, item in enumerate(EVAL_SET):
    query = item["question"]
    ground_truth = item["ground_truth"]
    logger.info(f"\n[{i+1}/{len(EVAL_SET)}] Processing: {query[:50]}...")

    for config in CONFIGS:
        config_name = config["name"]
        model_name = config["model"]
        use_rag = config["use_rag"]

        if model_name == "fine_tuned":
            agent.answer_generator = local_medgemma_generator
        else:
            current_generator = factory.create_generator(model_name)
            agent.answer_generator = current_generator

        answer = ""
        contexts = []

        try:
            if use_rag:
                answer, contexts = agent.ask(query)
            else:
                raw_prompt = f"""You are an expert clinical diagnostic assistant.
                Answer the user's question directly.
                **USER QUESTION:**
                {query}
                **ASSISTANT ANSWER:**"""

                gen = agent.answer_generator
                if hasattr(gen, "generate"):
                    answer = gen.generate(raw_prompt)
                elif hasattr(gen, "invoke"):
                    answer = gen.invoke(raw_prompt).content
                else:
                    answer = str(gen(raw_prompt))
                contexts = []

        except Exception as e:
            logger.error(f"Error in {config_name}: {e}")
            answer = "Error generating response."

        all_results.append({
            "question": query,
            "ground_truth": ground_truth,
            "answer": answer,
            "contexts": contexts,
            "type": config_name
        })

logger.info("Generation complete. Starting RAGAS evaluation...")

eval_dataset = Dataset.from_list(all_results)
metrics_to_run = [faithfulness, answer_relevancy, context_precision, context_recall, answer_correctness]
evaluation_config = RunConfig(max_workers=4, timeout=240)

result = evaluate(
    eval_dataset,
    metrics=metrics_to_run,
    llm=judge_llm,
    embeddings=hf_embeddings,
    run_config=evaluation_config
)

df = result.to_pandas()
df['type'] = [res['type'] for res in all_results]
df['question'] = [res['question'] for res in all_results]
df['answer'] = [res['answer'] for res in all_results]
df['ground_truth'] = [res['ground_truth'] for res in all_results]
df['contexts'] = [res['contexts'] for res in all_results]

print("\n--- Head of Results ---")
print(df.head())

output_csv = "/content/evaluation_results.csv"
df.to_csv(output_csv, index=False)
logger.info(f"Full results saved to {output_csv}")

print("\n 6-Way Comparison Summary (Averages)")
summary = df.groupby('type').mean(numeric_only=True)
print(summary)
summary.to_csv("/content/evaluation_summary.csv")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/usr/local/lib/python3.12/dist-packages/sentence_transformers/SentenceTransformer.py:204: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v4 of SentenceTransformers.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

==((====))==  Unsloth 2026.1.4: Fast Gemma3 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Gemma3 does not support SDPA - switching to fast eager.


model.safetensors:   0%|          | 0.00/4.12G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

Evaluating:   0%|          | 0/300 [00:00<?, ?it/s]


--- Head of Results ---
                                          user_input  \
0  A 32-year-old patient presents to the emergenc...   
1  A 32-year-old patient presents to the emergenc...   
2  A 32-year-old patient presents to the emergenc...   
3  A 32-year-old patient presents to the emergenc...   
4  A 32-year-old patient presents to the emergenc...   

                                  retrieved_contexts  \
0  [32-Year-Old Woman from Nigeria With Jaundice ...   
1                                                 []   
2  [32-Year-Old Woman from Nigeria With Jaundice ...   
3                                                 []   
4  [32-Year-Old Woman from Nigeria With Jaundice ...   

                                            response  \
0  Based on the information provided, the suspect...   
1  Based on the clinical presentation, the diagno...   
2  The patient's presentation strongly suggests *...   
3  Based on the patient's presentation, travel hi...   
4  The final diagnosi

DEMO ZONE

DEMO ZONE

In [ ]:
eval_set_content = r"""
EVAL_SET = [
    {
        "question": "An NS1 test is performed on a patient, and detects high concentrations of NS1 protein within their blood. A Reverse Transcriptase PCR (RT-PCR) test also reveals dengue virus RNA in the blood. What is this patient's initial diagnosis?",
        "ground_truth": "Dengue fever. The presense of high concentration of NS1 (Non-Structural Protein 1) protein in the blood suggests Dengue Virus, as this the protein most commonly produced by the dengue virus. The highly sensitive Reverse Transcriptase PCR (RT-PCR) test confirms this further, by confirming the presense of the viral Dengue RNA."
    }
]
"""

with open("evaluation_set.py", "w") as f:
    f.write(eval_set_content)

print("Evaluation set created successfully.")

Evaluation set created successfully.


In [ ]:
import os
import pandas as pd
import asyncio
import logging
import warnings
import importlib
import config
from google.colab import userdata

try:
    SECURE_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    SECURE_KEY = os.environ.get('GEMINI_API_KEY')

if not SECURE_KEY:
    import config
    SECURE_KEY = getattr(config, 'GEMINI_API_KEY', None)

if not SECURE_KEY:
    raise ValueError("CRITICAL: GEMINI_API_KEY is missing. Check Colab Secrets (Key symbol on the left).")

os.environ["GOOGLE_API_KEY"] = SECURE_KEY
os.environ["GEMINI_API_KEY"] = SECURE_KEY


import config
config.GEMINI_API_KEY = SECURE_KEY
importlib.reload(config)


from evaluation_set import EVAL_SET
from ragas import evaluate
from ragas.run_config import RunConfig
from datasets import Dataset
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from unsloth import FastLanguageModel


warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("google_genai._api_client").setLevel(logging.ERROR)


try:
    from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall, answer_correctness
except ImportError:
    from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall, answer_correctness


logger.info("Re-Initializing RAGAgent with verified API Key...")
agent = RAGAgent()
agent.genai_client = genai.Client(api_key=SECURE_KEY)
agent.generator_factory.genai_client = agent.genai_client
agent.answer_generator = agent.generator_factory.create_generator(agent.GENERATOR_MODEL_NAME)

factory = agent.generator_factory


judge_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-pro",
    temperature=0,
    max_retries=10,
    google_api_key=SECURE_KEY
)

EMBEDDING_MODEL_NAME = 'pritamdeka/S-BioBert-snli-multinli-stsb'
hf_embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={'token': config.HUGGINGFACETOKEN}
)

logger.info("Loading Fine-Tuned MedGemma...")
ft_model_path = "lora_model"
ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name = ft_model_path,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(ft_model)

class LocalMedGemmaWrapper:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer
        self.tok = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
        self.model_name = "medgemma_finetuned"

    def generate(self, prompt: str):
        formatted_prompt = f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
        inputs = self.tok(formatted_prompt, return_tensors="pt").to("cuda")
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=300,
            use_cache=True,
            pad_token_id=self.tok.eos_token_id
        )
        return self.tok.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

local_medgemma_generator = LocalMedGemmaWrapper(ft_model, ft_tokenizer)


CONFIGS = [
    {"name": "Gemini_RAG",        "model": "gemini-2.5-pro",       "use_rag": True},
    {"name": "Gemini_Raw",        "model": "gemini-2.5-pro",       "use_rag": False},
    {"name": "MedGemma_RAG",      "model": "google/medgemma-4b-it",      "use_rag": True},
    {"name": "MedGemma_Raw",      "model": "google/medgemma-4b-it",      "use_rag": False},
    {"name": "MedGemma_Finetuned_RAG", "model": "fine_tuned",            "use_rag": True},
    {"name": "MedGemma_Finetuned_Raw", "model": "fine_tuned",            "use_rag": False},
]

all_results = []
logger.info(f"Starting 6-Way Evaluation over {len(EVAL_SET)} questions...")

for i, item in enumerate(EVAL_SET):
    query = item["question"]
    ground_truth = item["ground_truth"]
    logger.info(f"\n[{i+1}/{len(EVAL_SET)}] Processing: {query[:50]}...")

    for config in CONFIGS:
        config_name = config["name"]
        model_name = config["model"]
        use_rag = config["use_rag"]

        if model_name == "fine_tuned":
            agent.answer_generator = local_medgemma_generator
        else:

            current_generator = factory.create_generator(model_name)
            agent.answer_generator = current_generator

        answer = ""
        contexts = []

        try:
            if use_rag:
                answer, contexts = agent.ask(query)
            else:
                raw_prompt = f"""You are an expert clinical diagnostic assistant.
                Answer the user's question directly.
                **USER QUESTION:**
                {query}
                **ASSISTANT ANSWER:**"""

                gen = agent.answer_generator
                if hasattr(gen, "generate"):
                    answer = gen.generate(raw_prompt)
                elif hasattr(gen, "invoke"):
                    answer = gen.invoke(raw_prompt).content
                else:
                    answer = str(gen(raw_prompt))
                contexts = []

        except Exception as e:
            logger.error(f"Error in {config_name}: {e}")
            answer = "Error generating response."

        all_results.append({
            "question": query,
            "ground_truth": ground_truth,
            "answer": answer,
            "contexts": contexts,
            "type": config_name
        })

logger.info("Generation complete. Starting RAGAS evaluation...")

eval_dataset = Dataset.from_list(all_results)
metrics_to_run = [faithfulness, answer_relevancy, context_precision, context_recall, answer_correctness]
evaluation_config = RunConfig(max_workers=4, timeout=240)

result = evaluate(
    eval_dataset,
    metrics=metrics_to_run,
    llm=judge_llm,
    embeddings=hf_embeddings,
    run_config=evaluation_config
)

df = result.to_pandas()
df['type'] = [res['type'] for res in all_results]
df['question'] = [res['question'] for res in all_results]
df['answer'] = [res['answer'] for res in all_results]
df['ground_truth'] = [res['ground_truth'] for res in all_results]
df['contexts'] = [res['contexts'] for res in all_results]

print("\n--- Head of Results ---")
print(df.head())

output_csv = "/content/demo_evaluation_results.csv"
df.to_csv(output_csv, index=False)
logger.info(f"Full results saved to {output_csv}")

print("\n 6-Way Comparison Summary (Averages)")
summary = df.groupby('type').mean(numeric_only=True)
print(summary)
summary.to_csv("/content/demo_evaluation_summary.csv")

/usr/local/lib/python3.12/dist-packages/sentence_transformers/SentenceTransformer.py:204: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v4 of SentenceTransformers.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

==((====))==  Unsloth 2026.1.4: Fast Gemma3 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Gemma3 does not support SDPA - switching to fast eager.


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]


--- Head of Results ---
                                          user_input  \
0  An NS1 test is performed on a patient, and det...   
1  An NS1 test is performed on a patient, and det...   
2  An NS1 test is performed on a patient, and det...   
3  An NS1 test is performed on a patient, and det...   
4  An NS1 test is performed on a patient, and det...   

                                  retrieved_contexts  \
0  [Another differential diagnosis to consider in...   
1                                                 []   
2  [Another differential diagnosis to consider in...   
3                                                 []   
4  [Another differential diagnosis to consider in...   

                                            response  \
0  Based on the laboratory results, the patient's...   
1  Based on these results, the patient's initial ...   
2  Based on the provided information, the patient...   
3  The patient's initial diagnosis is **Dengue Vi...   
4                    